# Einführung in Machine Learning – Schnelldurchgang

![](images/02-iris.png)

    Iris setosa                                                  Iris virginica                                              Iris versicolor

Quelle: [Wikipedia](https://de.wikipedia.org/wiki/Portal:Statistik/Datens%C3%A4tze#Iris)
- - -

Ein kompakter Vergleich von **k-Nearest Neighbours (KNN)** auf dem Iris-Datensatz: Warum die **Auswahl der Features** oft wichtiger ist als ein komplexeres Modell.

Ablauf: **Init → Training → 1. Falsche Features (Sepal) → 2. Richtige Features (Petal)**.

Der Vergleich verwendet einen festen, nach Klassen geschichteten Split (**60 % Training / 40 % Test**), damit die Ergebnisse reproduzierbar sind.

## Init

Wir laden den [Iris-Datensatz](https://scikit-learn.org/stable/auto_examples/datasets/plot_iris_dataset.html), bereiten die Daten vor und definieren eine wiederverwendbare Funktion zum Zeichnen der **Decision Boundaries**.

Benötigt werden **NumPy**, **Matplotlib** und **scikit-learn** im aktiven Jupyter-Kernel. Falls nötig, einmalig in einer separaten Zelle installieren: `%pip install numpy matplotlib scikit-learn`.

In [ ]:
%pip install numpy matplotlib scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn import __version__ as sklearn_version
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

print(f"scikit-learn: {sklearn_version}")
print(f"Datensatz: {X.shape[0]} Beobachtungen, {X.shape[1]} Features, {len(iris.target_names)} Klassen")
print("Features:", ", ".join(iris.feature_names))
print(f"Training: {len(X_train)} Beobachtungen | Test: {len(X_test)} Beobachtungen")
print(f"Erste Beobachtung: {X[0]} → {iris.target_names[y[0]]}")

In [ ]:
# Plot-Helfer: ein gemeinsames Koordinatensystem je Feature-Paar.
BG_COLORS = ListedColormap(["#f8cece", "#d5efcd", "#d5ddff"])
POINT_COLORS = ListedColormap(["#c0392b", "#247b35", "#335fba"])


def make_mesh(data, n=180):
    x_min, y_min = data.min(axis=0) - 0.3
    x_max, y_max = data.max(axis=0) + 0.3
    return np.meshgrid(np.linspace(x_min, x_max, n), np.linspace(y_min, y_max, n))


def plot_regions(ax, model, features, labels, feature_names, title, mesh):
    xx, yy = mesh
    predictions = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, predictions, levels=[-0.5, 0.5, 1.5, 2.5], cmap=BG_COLORS)
    ax.scatter(
        features[:, 0], features[:, 1], c=labels, cmap=POINT_COLORS,
        vmin=-0.5, vmax=2.5, s=34, edgecolors="black", linewidths=0.35
    )
    ax.set(xlabel=feature_names[0], ylabel=feature_names[1], title=title)
    ax.set_xlim(xx.min(), xx.max())
    ax.set_ylim(yy.min(), yy.max())
    ax.grid(alpha=0.15)


def print_scores(name, model, train_X, test_X):
    train_acc = model.score(train_X, y_train)
    test_acc = model.score(test_X, y_test)
    print(f"{name:<23} Training: {train_acc:6.1%} | Test: {test_acc:6.1%}")
    return train_acc, test_acc

## Training

Zuerst trainieren wir einen KNN-Klassifikator mit **allen vier Iris-Features** und **K=1** (ein nächster Nachbar). Wir vergleichen die Treffergenauigkeit auf Trainings- und Testdaten und betrachten zwei Beispielvorhersagen.

**Wichtig:** Ein hoher Trainings-Score allein sagt noch nicht aus, ob das Modell auf unbekannte Daten generalisiert.

In [ ]:
knn_all = KNeighborsClassifier(n_neighbors=1).fit(X_train, y_train)
all_scores = print_scores("Alle 4 Features, K=1", knn_all, X_train, X_test)

sample_id = 32
sample_features, sample_label = X_test[sample_id], y_test[sample_id]
predicted_label = knn_all.predict([sample_features])[0]
manual_features = [[6.3, 2.7, 5.5, 1.5]]
manual_label = knn_all.predict(manual_features)[0]

print(f"\nTestbeispiel {sample_id}: {sample_features}")
print(f"Ist: {iris.target_names[sample_label]} | Prognose: {iris.target_names[predicted_label]}")
print(f"Freies Beispiel {manual_features[0]} → {iris.target_names[manual_label]}")

## 1. Falsche Features – Sepal

Jetzt verwenden wir **nur die beiden Kelchblatt-Masse** (`sepal length`, `sepal width`). Für die Unterscheidung der drei Iris-Arten sind sie vergleichsweise wenig aussagekräftig.

Wir vergleichen **K=1** (komplexe, kleinteilige Decision Boundaries; Overfitting-Risiko) mit **K=10** (glattere Grenzen). In den Grafiken stehen die **Zeilen** für K und die **Spalten** für Training bzw. Test. Die Hintergrundfarbe zeigt die vorhergesagte Klasse, die Punkte zeigen die tatsächliche Klasse.

In [ ]:
X_train_sepal, X_test_sepal = X_train[:, :2], X_test[:, :2]
sepal_models = {
    k: KNeighborsClassifier(n_neighbors=k).fit(X_train_sepal, y_train)
    for k in (1, 10)
}
sepal_scores = {
    k: print_scores(f"Sepal, K={k}", model, X_train_sepal, X_test_sepal)
    for k, model in sepal_models.items()
}

mesh = make_mesh(X[:, :2])  # identische Achsen für alle vier Darstellungen
fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True)
for row, (k, model) in enumerate(sepal_models.items()):
    for col, (name, features, labels) in enumerate([
        ("Training", X_train_sepal, y_train),
        ("Test", X_test_sepal, y_test),
    ]):
        plot_regions(axes[row, col], model, features, labels,
                     iris.feature_names[:2], f"Sepal · K={k} · {name}", mesh)
fig.suptitle("Sepal: K=1 passt sich stark an, K=10 glättet – geringe Trennschärfe bleibt", fontsize=13)
plt.tight_layout()
plt.show()

## 2. Richtige Features – Petal

Wir ersetzen die Sepal- durch die **Kronblatt-Masse** (`petal length`, `petal width`) und verwenden wieder **K=10**. Der Algorithmus und der Train/Test-Split bleiben gleich: Es ändert sich nur die **Feature-Auswahl**.

Vergleiche insbesondere den **Test-Score** mit dem Sepal-Modell. Die beiden Diagramme zeigen die Decision Boundaries für Training und Test bei identischem Achsenbereich.

In [ ]:
X_train_petal, X_test_petal = X_train[:, 2:], X_test[:, 2:]
knn_petal = KNeighborsClassifier(n_neighbors=10).fit(X_train_petal, y_train)
petal_scores = print_scores("Petal, K=10", knn_petal, X_train_petal, X_test_petal)

mesh = make_mesh(X[:, 2:])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)
for ax, (name, features, labels) in zip(axes, [
    ("Training", X_train_petal, y_train),
    ("Test", X_test_petal, y_test),
]):
    plot_regions(ax, knn_petal, features, labels,
                 iris.feature_names[2:], f"Petal · K=10 · {name}", mesh)
fig.suptitle("Petal: gute Trennung der Klassen mit nur zwei Features", fontsize=13)
plt.tight_layout()
plt.show()

print("\nTest-Accuracy im Vergleich:")
for name, accuracy in [
    ("Alle 4 Features, K=1", all_scores[1]),
    ("Sepal, K=1", sepal_scores[1][1]),
    ("Sepal, K=10", sepal_scores[10][1]),
    ("Petal, K=10", petal_scores[1]),
]:
    print(f"  {name:<23} {accuracy:6.1%}")

**Fazit:** Bei diesem Datensatz trennen die beiden Petal-Features die Iris-Klassen deutlich besser als die beiden Sepal-Features. Mit denselben Trainings- und Testdaten kann bereits die Wahl der Eingangsmerkmale die Generalisierung stark verbessern. Die Bewertung auf separaten Testdaten hilft, Overfitting zu erkennen.